# Sen1Floods11, Sentinel-1 only: weak-label pretraining + multi-encoder U-Net ensemble

One notebook, run top to bottom on a Kaggle T4 (Internet ON, GPU T4).

What it does
1. Gets the data (public bucket) and the official split (train / valid / test / Bolivia).
2. Stage A: pretrains on the weakly labelled S1 chips, **with every valid, test and Bolivia chip removed by ID, and all Bolivia weak chips removed by event** (this is the leak fix).
3. Stage B: fine-tunes on the 252 hand-labelled training chips, picks the best epoch on valid.
4. Ensembles encoders x seeds with flip TTA, picks the threshold on valid only, reports water-class IoU on test and Bolivia with bootstrap CIs.
5. Optional ablation: same model without weak pretraining.

Set `SMOKE=True` in the config cell for a ~5 minute dry run first.

Metric: pooled IoU of the water class (label 1), pixels labelled -1 ignored. Compare to published numbers only after checking their metric and label setup.

In [ ]:
import os, sys, glob, re, json, time, random, shutil, subprocess, warnings
warnings.filterwarnings("ignore")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "rasterio"], check=False)

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import rasterio
import segmentation_models_pytorch as smp

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("device:", DEV, "| torch", torch.__version__, "| smp", smp.__version__)

In [ ]:
CFG = dict(
    SMOKE=False,                      # True = tiny dry run to catch path/API errors
    ENCODERS=["resnet34", "resnext50_32x4d", "mit_b2"],   # encoders that fail to build are skipped
    SEEDS=[0, 1, 2],
    USE_WEAK=True,
    ABLATE_NOWEAK=True,               # also train first encoder / first seed without weak pretraining
    WEAK_EPOCHS=10, WEAK_REPS=2,      # passes over weak chips (each pass = 1 random crop per chip x reps)
    FT_EPOCHS=80, FT_REPS=4,
    CROP=256, BS=24,
    LR_WEAK=5e-4, LR_FT=2e-4, WD=1e-4,
    WORKERS=3,
    OUT="/kaggle/working/out",
)
if CFG["SMOKE"]:
    CFG.update(ENCODERS=CFG["ENCODERS"][:1], SEEDS=[0], WEAK_EPOCHS=1, WEAK_REPS=1, FT_EPOCHS=5, FT_REPS=1, ABLATE_NOWEAK=False)
os.makedirs(CFG["OUT"], exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ---------- data ----------
ROOT = "/kaggle/working/s1f11"
B = "gs://sen1floods11/v1.1"
def sh(c, show=True):
    if show: print("$", c)
    return subprocess.run(c, shell=True, capture_output=True, text=True)

if not glob.glob(f"{ROOT}/S1Hand/*.tif"):
    if shutil.which("gsutil") is None:
        sh("pip install -q gsutil")
    os.makedirs(ROOT, exist_ok=True)
    sh(f"gsutil -m -q cp -r {B}/splits/flood_handlabeled {ROOT}/")
    for d in ["S1Hand", "LabelHand"]:
        r = sh(f"gsutil -m -q cp -r {B}/data/flood_events/HandLabeled/{d} {ROOT}/")
        print(r.stderr[-300:])
    ls = sh(f"gsutil ls {B}/data/flood_events/WeaklyLabeled/").stdout
    print(ls)
    weak_dirs = [l.rstrip("/").split("/")[-1] for l in ls.split() if l.strip()]
    lab = [d for d in weak_dirs if "label" in d.lower() and "s2" in d.lower()] or [d for d in weak_dirs if "label" in d.lower()]
    print("weak dirs:", weak_dirs, "-> label dir:", lab[:1])
    for d in ["S1Weak"] + lab[:1]:
        r = sh(f"gsutil -m -q cp -r {B}/data/flood_events/WeaklyLabeled/{d} {ROOT}/")
        print(r.stderr[-300:])

def find_csv(name):
    p = glob.glob(f"{ROOT}/**/{name}", recursive=True)
    assert p, f"missing {name}; check download output above"
    return p[0]

def read_split(name):
    df = pd.read_csv(find_csv(name), header=None)
    return [(f"{ROOT}/S1Hand/{a}", f"{ROOT}/LabelHand/{b}") for a, b in zip(df[0], df[1])]

SPLIT = {k: read_split(f"flood_{k}_data.csv") for k in ["train", "valid", "test", "bolivia"]}
SPLIT["bol"] = SPLIT.pop("bolivia")
print({k: len(v) for k, v in SPLIT.items()})
assert all(os.path.exists(a) and os.path.exists(b) for v in SPLIT.values() for a, b in v), "hand-labelled files missing"

# ---------- weak labels with leak fix ----------
def cid(path):  # chip id e.g. Ghana_103272
    return re.sub(r"_(S1Hand|S1Weak|LabelHand)\.tif$", "", os.path.basename(path))

heldout_ids = {cid(a) for k in ["valid", "test", "bol"] for a, _ in SPLIT[k]}
WEAK = []
if CFG["USE_WEAK"]:
    weak_lab_dirs = [d for d in os.listdir(ROOT) if "label" in d.lower() and "weak" in d.lower()]
    assert weak_lab_dirs, f"no weak label dir in {os.listdir(ROOT)}"
    wl = f"{ROOT}/{weak_lab_dirs[0]}"
    lab_index = {}
    for p in glob.glob(f"{wl}/*.tif"):
        lab_index[re.sub(r"_[A-Za-z0-9]*Label[A-Za-z0-9]*\.tif$", "", os.path.basename(p))] = p
    n_all = n_leak = 0
    for p in sorted(glob.glob(f"{ROOT}/S1Weak/*.tif")):
        i = cid(p); n_all += 1
        if i in heldout_ids or i.startswith("Bolivia_"):
            n_leak += 1; continue
        if i in lab_index: WEAK.append((p, lab_index[i]))
    print(f"weak chips: {n_all} found, {n_leak} removed (held-out ids / Bolivia), {len(WEAK)} usable (label dir: {weak_lab_dirs[0]})")
    if CFG["SMOKE"]: WEAK = WEAK[:200]
    assert len(WEAK) > 100, "too few weak pairs; check filename pattern printed above"

In [ ]:
# ---------- preprocessing / datasets ----------
def read_tif(p):
    with rasterio.open(p) as s:
        return s.read().astype(np.float32)

def prep(a, shift=0.0):
    a = np.nan_to_num(a, nan=-30.0, posinf=0.0, neginf=-30.0)
    a = np.clip(a[:2] + shift, -30.0, 5.0)
    return np.stack([a[0], a[1], a[0] - a[1]]).astype(np.float32)

def load_set(items):
    X = np.stack([prep(read_tif(a)) for a, _ in items])
    Y = np.stack([read_tif(b)[0] for _, b in items]).astype(np.int8)
    return X, Y

H = {k: load_set(v) for k, v in SPLIT.items()}
MEAN = H["train"][0].mean((0, 2, 3), keepdims=True)[0].astype(np.float32)
STD = H["train"][0].std((0, 2, 3), keepdims=True)[0].astype(np.float32) + 1e-6
norm = lambda x: (x - MEAN) / STD
print("mean", MEAN.ravel(), "std", STD.ravel())

def augment(x, y, crop):
    _, h, w = x.shape
    if crop and h > crop:
        i, j = np.random.randint(0, h - crop + 1), np.random.randint(0, w - crop + 1)
        x, y = x[:, i:i + crop, j:j + crop], y[i:i + crop, j:j + crop]
    if np.random.rand() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
    if np.random.rand() < .5: x, y = x[:, ::-1], y[::-1]
    k = np.random.randint(4)
    if k: x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)
    return np.ascontiguousarray(x), np.ascontiguousarray(y)

class ArrDS(torch.utils.data.Dataset):
    def __init__(self, X, Y, reps, crop): self.X, self.Y, self.reps, self.crop = X, Y, reps, crop
    def __len__(self): return len(self.X) * self.reps
    def __getitem__(self, i):
        i %= len(self.X)
        x, y = augment(norm(self.X[i]), self.Y[i], self.crop)
        return torch.from_numpy(x), torch.from_numpy(y.astype(np.float32))[None]

class WeakDS(torch.utils.data.Dataset):
    def __init__(self, items, reps, crop): self.items, self.reps, self.crop = items, reps, crop
    def __len__(self): return len(self.items) * self.reps
    def __getitem__(self, i):
        a, b = self.items[i % len(self.items)]
        x = prep(read_tif(a), shift=np.random.uniform(-1, 1)); y = read_tif(b)[0]
        x, y = augment(norm(x), y, self.crop)
        return torch.from_numpy(x), torch.from_numpy(y.astype(np.float32))[None]

def loader(ds, shuffle=True):
    return torch.utils.data.DataLoader(ds, batch_size=CFG["BS"], shuffle=shuffle, num_workers=CFG["WORKERS"],
                                       pin_memory=True, drop_last=shuffle, persistent_workers=CFG["WORKERS"] > 0)

In [ ]:
# ---------- model / loss / metrics ----------
def make_model(enc):
    return smp.Unet(enc, encoder_weights="imagenet", in_channels=3, classes=1).to(DEV)

def masked_loss(logit, y):
    m = (y >= 0).float(); t = (y == 1).float()
    bce = (F.binary_cross_entropy_with_logits(logit, t, reduction="none") * m).sum() / m.sum().clamp(min=1)
    p = torch.sigmoid(logit) * m
    inter = (p * t).sum((1, 2, 3)); den = p.sum((1, 2, 3)) + (t * m).sum((1, 2, 3))
    return bce + (1 - ((2 * inter + 1) / (den + 1)).mean())

def counts(prob, y, thr=0.5):
    m = y >= 0; pr = (prob >= thr) & m; t = (y == 1) & m
    return (pr & t).sum((1, 2)), (pr & ~t).sum((1, 2)), (~pr & t & m).sum((1, 2))   # tp, fp, fn per chip

def iou_from(tp, fp, fn): return tp.sum() / max(tp.sum() + fp.sum() + fn.sum(), 1)

@torch.no_grad()
def predict(model, X, tta=True, bs=8):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        x = torch.from_numpy(norm_batch(X[i:i + bs])).to(DEV)
        ps = []
        flips = [[], [3], [2], [2, 3]] if tta else [[]]
        for f in flips:
            xi = torch.flip(x, f) if f else x
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
                o = torch.sigmoid(model(xi).float())
            ps.append(torch.flip(o, f) if f else o)
        out.append(torch.stack(ps).mean(0)[:, 0].cpu().numpy().astype(np.float16))
    return np.concatenate(out)

def norm_batch(x): return ((x - MEAN[None]) / STD[None]).astype(np.float32)

def val_iou(model):
    p = predict(model, H["valid"][0], tta=False)
    return float(iou_from(*counts(p.astype(np.float32), H["valid"][1])))

def fit(model, dl, epochs, lr, tag="", val=False):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(dl), pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda")
    best, best_sd = -1, None; t0 = time.time()
    for ep in range(epochs):
        model.train(); tot = 0
        for x, y in dl:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
                out = model(x)
            loss = masked_loss(out.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        msg = f"[{tag}] ep {ep + 1}/{epochs} loss {tot / len(dl):.4f} ({time.time() - t0:.0f}s)"
        if val and ((ep + 1) % 5 == 0 or ep == epochs - 1):
            v = val_iou(model); msg += f" valid IoU {v:.4f}"
            if v > best: best, best_sd = v, {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        print(msg)
    if best_sd is not None: model.load_state_dict(best_sd)
    return best

In [ ]:
# ---------- run all (encoder x seed) ----------
RUNS = []
def run(enc, seed, use_weak):
    set_seed(seed)
    try: model = make_model(enc)
    except Exception as e:
        print(f"skip {enc}: {type(e).__name__}: {str(e)[:200]}"); return None
    tag = f"{enc}|s{seed}|{'weak' if use_weak else 'noweak'}"
    if use_weak and WEAK:
        fit(model, loader(WeakDS(WEAK, CFG["WEAK_REPS"], CFG["CROP"])), CFG["WEAK_EPOCHS"], CFG["LR_WEAK"], tag + "|A")
    best = fit(model, loader(ArrDS(*H["train"], CFG["FT_REPS"], CFG["CROP"])), CFG["FT_EPOCHS"], CFG["LR_FT"], tag + "|B", val=True)
    probs = {k: predict(model, H[k][0]) for k in ["valid", "test", "bol"]}
    r = dict(enc=enc, seed=seed, weak=use_weak, best_valid=best, probs=probs)
    r["test_iou"] = float(iou_from(*counts(probs["test"].astype(np.float32), H["test"][1])))
    r["bol_iou"] = float(iou_from(*counts(probs["bol"].astype(np.float32), H["bol"][1])))
    print(f"==> {tag}: valid {best:.4f} | test {r['test_iou']:.4f} | bolivia {r['bol_iou']:.4f}")
    del model; torch.cuda.empty_cache()
    return r

for enc in CFG["ENCODERS"]:
    for seed in CFG["SEEDS"]:
        r = run(enc, seed, CFG["USE_WEAK"])
        if r: RUNS.append(r)
ABL = []
if CFG["ABLATE_NOWEAK"] and CFG["USE_WEAK"] and RUNS:
    r = run(RUNS[0]["enc"], RUNS[0]["seed"], False)
    if r: ABL.append(r)
assert RUNS, "no model trained"

In [ ]:
# ---------- ensemble, threshold on valid only, bootstrap CI ----------
def ens(runs, split): return np.mean([r["probs"][split].astype(np.float32) for r in runs], 0)

def best_thr(runs):
    pv = ens(runs, "valid"); grid = np.arange(0.30, 0.71, 0.02)
    return float(grid[int(np.argmax([iou_from(*counts(pv, H["valid"][1], t)) for t in grid]))])

def boot_ci(prob, y, thr, n=2000, seed=0):
    tp, fp, fn = counts(prob, y, thr); rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(tp), len(tp)); vals.append(iou_from(tp[i], fp[i], fn[i]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def report(name, runs):
    thr = best_thr(runs); row = dict(model=name, n_models=len(runs), thr=thr)
    for sp in ["valid", "test", "bol"]:
        p = ens(runs, sp); row[sp] = float(iou_from(*counts(p, H[sp][1], thr)))
        if sp != "valid": row[sp + "_ci95"] = boot_ci(p, H[sp][1], thr)
    return row

rows = []
for enc in dict.fromkeys(r["enc"] for r in RUNS):
    sub = [r for r in RUNS if r["enc"] == enc]
    rows.append(report(f"{enc} x{len(sub)} seeds", sub))
rows.append(report(f"ALL ensemble ({len(RUNS)} models)", RUNS))
if ABL:
    rows.append(report(f"{ABL[0]['enc']} s{ABL[0]['seed']} NO weak pretrain", ABL))
    same = [r for r in RUNS if r["enc"] == ABL[0]["enc"] and r["seed"] == ABL[0]["seed"]]
    rows.append(report(f"{ABL[0]['enc']} s{ABL[0]['seed']} WITH weak pretrain", same))

df = pd.DataFrame(rows)
pd.set_option("display.width", 220, "display.max_colwidth", 60)
print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

singles = pd.DataFrame([{k: r[k] for k in ["enc", "seed", "weak", "best_valid", "test_iou", "bol_iou"]} for r in RUNS + ABL])
print("\nsingle models:\n", singles.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

df.to_csv(f"{CFG['OUT']}/ensemble_results.csv", index=False); singles.to_csv(f"{CFG['OUT']}/single_models.csv", index=False)
np.savez_compressed(f"{CFG['OUT']}/ens_probs.npz", **{sp: ens(RUNS, sp).astype(np.float16) for sp in ["valid", "test", "bol"]})
json.dump(dict(cfg=CFG, mean=MEAN.ravel().tolist(), std=STD.ravel().tolist()), open(f"{CFG['OUT']}/config.json", "w"), indent=1)
print("saved to", CFG["OUT"])

## How to read the output
- The `ALL ensemble` row, test IoU, is the number to compare with published Sentinel-1 results (water class, hand-labelled test split). Report the CI.
- The NO / WITH weak pretrain pair is your ablation for the paper. It is a single seed, so say so.
- Published figures I could verify: DeepSARFlood reports 0.72 IoU on the Sen1Floods11 test set (S1 only, ViT ensembles). Other papers use other metrics or labels (for example mIoU, flood-only class, optical bands), so check before comparing.
- Do not tune anything on test or Bolivia. The threshold here comes from valid only.